# Aula 03 - Hiperparâmetros em RN - Fashion MNIST (19.08.2026)

# Michel Cirilo de Oliveira / Eric Berto

# Setup

In [13]:
# Importar bibliotecas utilizadas no notebook

import numpy as np                              # Computação numérica
import matplotlib.pyplot as plt                 # Geração de gráficos
import time                                     # Medição do tempo de treinamento

from sklearn.model_selection import train_test_split  # Separação entre treino e teste

import tensorflow as tf                         # Construção e treinamento de redes neurais
from tensorflow import keras                    # API de alto nível do TensorFlow
from tensorflow.keras import layers             # Construção das camadas da rede

# Definir semente para tornar os experimentos reproduzíveis

np.random.seed(42)
tf.random.set_seed(42)

# Exibir gráficos no notebook

%matplotlib inline

### Carregar o conjunto de dados Fashion-MNIST via Keras

In [14]:
# Carregar conjunto de dados Fashion-MNIST diretamente do Keras
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Combine os dados para exibir informações básicas, se necessário, ou use os conjuntos separados
# Para replicar a estrutura anterior, vamos criar um dicionário 'dados'
# Nota: as imagens aqui já são arrays numpy, então não precisam de .npz

dados_keras = {
    'X_train': X_train,
    'y_train': y_train,
    'X_test': X_test,
    'y_test': y_test
}

# Exibir informações básicas sobre o conjunto de dados

print(f"Quantidade de instâncias: {dados_keras['X_train'].shape[0] + dados_keras['X_test'].shape[0]}")
print(f"Dimensão das imagens: {dados_keras['X_train'].shape[1]} x {dados_keras['X_train'].shape[2]} pixels")
print(f"Quantidade de pixels por imagem: {dados_keras['X_train'].shape[1] * dados_keras['X_train'].shape[2]}")
print(f"Quantidade de classes: {len(np.unique(dados_keras['y_train']))}")

# Agora, você pode usar 'dados_keras' onde antes usava 'dados'
# Por exemplo, nas células seguintes, você precisará ajustar para usar X_train, y_train, etc., diretamente ou adaptar para usar 'dados_keras'.
# Para fins de compatibilidade imediata com o resto do notebook, vamos atribuir os resultados às variáveis esperadas:
X_train = dados_keras['X_train']
y_train = dados_keras['y_train']
X_test = dados_keras['X_test']
y_test = dados_keras['y_test']

Quantidade de instâncias: 70000
Dimensão das imagens: 28 x 28 pixels
Quantidade de pixels por imagem: 784
Quantidade de classes: 10


In [17]:
# Carregar conjunto de dados utilizado nos experimentos

dados = np.load("fashion_mnist.npz", allow_pickle=True)

# Exibir informações básicas sobre o conjunto de dados

print(f"Quantidade de instâncias: {dados['X_train'].shape[0] + dados['X_test'].shape[0]}")
print(f"Dimensão das imagens: {dados['X_train'].shape[1]} x {dados['X_train'].shape[2]} pixels")
print(f"Quantidade de pixels por imagem: {dados['X_train'].shape[1] * dados['X_train'].shape[2]}")
print(f"Quantidade de classes: {len(np.unique(dados['y_train']))}")

Quantidade de instâncias: 70000
Dimensão das imagens: 28 x 28 pixels
Quantidade de pixels por imagem: 784
Quantidade de classes: 10


In [18]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Definir nomes das classes do Fashion-MNIST

nomes_classes = ["T-shirt/top",    "Trouser",    "Pullover",    "Dress",    "Coat",    "Sandal",    "Shirt",    "Sneaker",    "Bag",    "Ankle boot"]

# Visualizar uma imagem de cada classe do conjunto de dados

plt.figure(figsize=(12, 5))

for classe in range(10):

    # Localizar a primeira imagem pertencente à classe
    indice = np.where(dados["y_train"] == classe)[0][0]

    plt.subplot(2, 5, classe + 1)
    plt.imshow(dados["X_train"][indice], cmap="gray")
    plt.title(f"{classe} - {nomes_classes[classe]}")
    plt.axis("off")

# Configurar o gráfico

plt.suptitle("Classes do Fashion-MNIST")
plt.tight_layout()

plt.show()

## Perguntas

1. Quantas variáveis de entrada e quantas classes existem?

R. O conjunto de dados possui 784 variáveis de entrada e 10 classes (tipos de vestuário)

2. O que representa cada variável de entrada neste conjunto de dados?

R. Cada variável de entrada representa o valor (intensidade) de um único pixel da imagem original (que tem dimensões de 28x28) quando transformada em um vetor unidimensional. Após a etapa de pré-processamento, o valor original (0 a 255) foi normalizado para uma escala entre 0 e 1.

3. Por que um problema com essa quantidade de variáveis e classes pode ser interessante para avaliar diferentes arquiteturas e hiperparâmetros de uma RN?

R. O Fashion-MNIST é suficientemente complexo para exigir uma rede neural capaz de aprender padrões não lineares (diferente de problemas trivialmente separáveis), mas possui um tamanho gerenciável (70.000 instâncias totais) que permite treinamentos rápidos e testes eficientes de várias arquiteturas, camadas e hiperparâmetros sem o alto custo computacional de imagens coloridas e de alta resolução.

# Split dos Dados

In [ ]:
# Preparar os dados para treinamento dos modelos

# Recuperar variáveis de entrada e variável alvo
X_train = dados["X_train"]
y_train = dados["y_train"]

X_test = dados["X_test"]
y_test = dados["y_test"]

# Transformar cada imagem 28x28 em um vetor com 784 variáveis de entrada
X_train = X_train.reshape(X_train.shape[0], 28 * 28)
X_test = X_test.reshape(X_test.shape[0], 28 * 28)

# Normalizar os valores dos pixels para o intervalo entre 0 e 1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Exibir quantidade de instâncias e variáveis de entrada
print(f"Treinamento: {len(X_train)} instâncias")
print(f"Teste: {len(X_test)} instâncias")
print(f"Variáveis de entrada: {X_train.shape[1]}")

# Construção e Treinamento da RN

In [ ]:
# Construir modelo de referência

model = keras.Sequential([
    layers.Input(shape=(784,)),                 # Dimensionalidade dos dados de entrada
    layers.Dense(128, activation="relu"),       # 1ª camada oculta: ReLU
    layers.Dense(10, activation="softmax")      # Saída: Softmax
])

# Configurar processo de treinamento

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),  # Otimizador Adam e taxa de aprendizado
    loss="sparse_categorical_crossentropy",                 # Função de perda
    metrics=["accuracy"]                                    # Métrica de avaliação
)

# Exibir arquitetura do modelo

model.summary()

## Perguntas

4. Quantas camadas ocultas possui a rede e quantos neurônios existem nessa camada?
R.   A rede possui 1 camada oculta contendo 128 neurônios.  
    layers.Dense(128, activation="relu"),       # 1ª camada oculta: ReLU
    layers.Dense(10, activation="softmax")      # Saída: Softmax

5. Qual função de ativação foi utilizada na camada oculta? Qual é seu papel?

R. Foi utilizada a função de ativação ReLU (relu). O seu papel é introduzir não-linearidade no modelo, permitindo que a rede aprenda mapeamentos complexos e ajudando a evitar o problema de desaparecimento do gradiente.

6. Por que a camada de saída possui 10 neurônios e utiliza a função Softmax?

R. A camada de saída tem 10 neurônios porque o conjunto de dados foi classificado em exatamente 10 categorias diferentes. A função Softmax transforma os valores numéricos gerados pela camada final em uma distribuição de probabilidades, onde a soma de todas as 10 saídas é exatamente 1 (ou 100%), permitindo ao modelo apontar a classe mais provável.

7. Qual função de perda foi escolhida e por que ela é adequada para este problema?

R. A função escolhida foi a sparse_categorical_crossentropy. Ela é a opção adequada para classificação multiclasse quando a variável alvo (label) está estruturada na forma de valores numéricos inteiros (como o range de 0 a 9) e não no formato de vetores one-hot encoding (variáveis dummy)

8. Qual otimizador e qual taxa de aprendizado foram definidos?

R. O otimizador definido foi o Adam, com uma taxa de aprendizado (learning rate) configurada para 0.001.

9. O que acontece se alterarmos `shape=(784,)` para outro valor? Por quê?

R. O código resultará em um erro de dimensão logo no início do treinamento. A camada Input deforma uma expectativa da estrutura dos dados, e como os arrays de imagens foram achatados (usando o método .reshape) para exatamente 28 * 28 = 784 posições independentes, os dados alimentados deixarão de encaixar na rede.

10. Quantos parâmetros treináveis possui a rede? Como esse valor é calculado a partir da quantidade de entradas, neurônios, pesos e biases?

R. A rede possui um total de 101.770 parâmetros treináveis. O cálculo é a soma dos pesos e vieses (biases) de cada camada:  Camada Oculta: (784 entradas × 128 neurônios) + 128 biases = 100.480 parâmetros.  Camada de Saída: (128 entradas da oculta × 10 neurônios) + 10 biases = 1.290 parâmetros.  Total: 100.480 + 1.290 = 101.770 parâmetros.  


In [ ]:
# Treinar modelo de referência

inicio = time.time()

historico = model.fit(
    X_train,
    y_train,
    epochs=20,             # Número de épocas
    batch_size=128,        # Tamanho do lote
    validation_split=0.20, # Utilizar 20% do treinamento para validação
    verbose=1
)

fim = time.time()

# Exibir tempo total de treinamento

print(f"\nTempo de treinamento: {fim - inicio:.2f} segundos")

## Perguntas

11. Qual tamanho de lote (batch size) foi utilizado?
R. O lote selecionado foi de 128 instâncias.

12. Considerando a quantidade de instâncias utilizadas no treinamento, aproximadamente quantas atualizações dos parâmetros ocorrem em cada época?
R. Foram fornecidas 60.000 instâncias para o método .fit(), porém houve a definição de validation_split=0.20, o que reserva 20% para a validação. Isso deixa 48.000 instâncias para o treino efetivo. Ao dividir essas 48.000 pelo lote de 128, chega-se a exatas 375 atualizações (steps) por época.

13. Quantas épocas foram utilizadas e o que uma época representa neste treinamento?
R. O modelo foi treinado por 20 épocas. Uma época representa um ciclo iterativo completo no qual todo o conjunto de dados de treinamento passa de ponta a ponta pela rede para a atualização dos parâmetros.

14. Qual é o papel do conjunto de validação utilizado durante o treinamento?
R. Os 20% de dados reservados servem para verificar a performance do modelo em dados inéditos ao final de cada época, monitorando a sua real capacidade de generalização e revelando quando o modelo começa a decorar os dados (overfitting).

15. Como os valores de loss e accuracy se comportaram ao longo das épocas?
R. Nas métricas de treino o progresso foi constante: a perda reduziu severamente (de 0.5727 até 0.1972) enquanto a precisão disparou (de 80.38% até 92.88%). Por outro lado, nas métricas de validação (val_loss e val_accuracy), ocorreu um pico rápido de otimização nas primeiras épocas e depois o modelo começou a estagnar, travando a precisão geral por volta de 88% e fazendo o erro começar a flutuar.

# Avaliação

In [ ]:
# Avaliar modelo no conjunto de teste

loss_teste, acuracia_teste = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

# Exibir resultados da avaliação

print(f"Loss no teste: {loss_teste:.4f}")
print(f"Acurácia no teste: {acuracia_teste:.4f}")

In [ ]:
# Visualizar evolução da função de perda durante o treinamento

plt.figure(figsize=(8, 5))

plt.plot(historico.history["loss"], label="Perda nos dados de treinamento")
plt.plot(historico.history["val_loss"], label="Perda nos dados de validação")

# Configurar o gráfico

plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Evolução da Função de Perda")
plt.legend()

plt.show()

16. O que representam as curvas de perda nos dados de treinamento e de validação?

R. Essas curvas medem as penalidades numéricas geradas pela margem de erro. A curva de treinamento demonstra o quanto o modelo aprende a mapear a resposta para as instâncias atuais. A de validação reflete o quanto desse padrão abstrato é aplicável ao mundo real em cenários ainda não testados.

17. Por que a perda nos dados de treinamento continua diminuindo ao longo das épocas?

R. Porque o otimizador foca continuamente em ajustar os pesos da rede para reduzir o gradiente das funções que compõem aquele dataset em específico. Se for ensinado vezes suficientes, o modelo eventualmente irá memorizar a base inteira (overfitting).

18. A partir de aproximadamente qual época a perda de validação deixa de apresentar melhora significativa?

R. Aproximadamente a partir da época 13 (que inclusive registrou a melhor perda, marcando 0.3391). Após esse ponto a curva congela o ganho da generalização e até mesmo aumenta gradativamente até o fim (0.3576 na época 20).


In [ ]:
# Identificar a época com menor perda de validação

melhor_epoca = np.argmin(historico.history["val_loss"]) + 1
melhor_val_loss = np.min(historico.history["val_loss"])

print(f"Melhor época: {melhor_epoca}")
print(f"Menor val_loss: {melhor_val_loss:.4f}")

# Uso do Modelo
Pegue imagens da internet e teste o modelo.

In [ ]:
import tensorflow.keras.utils as image

# Carregar e preparar uma nova imagem para classificação

imagem = image.load_img(
    "sample_data/bota.png",
    color_mode="grayscale",
    target_size=(28, 28)
)

imagem = image.img_to_array(imagem)
imagem = imagem / 255.0
imagem = imagem.reshape(1, 784)

# Realizar classificação utilizando o modelo treinado

probabilidades = model.predict(imagem, verbose=0)
classe_prevista = np.argmax(probabilidades)

print(f"Classe prevista: {nomes_classes[classe_prevista]}")
print(f"Probabilidade: {probabilidades[0][classe_prevista] * 100:.2f}%")

## Pergunta
19. Por que o modelo classificou incorretamente a imagem da bota como Bag?

R.: Provavelmente devido a divergências na padronização da imagem. O Fashion-MNIST é um banco com processamento homogêneo rigoroso, onde o plano de fundo é sempre uniformemente escuro e os itens focados e iluminados no centro. Se a imagem da bota recém baixada possuir um fundo claro, ruídos ou recortes diferentes dos 28x28 da base, a rede ativará pesos aleatórios sem sentido, os quais neste caso penderam acidentalmente para o grupo de bolsas.

20. Como o modelo pode apresentar probabilidade de 100% e ainda assim estar errado?

R.: A saída da função Softmax gera uma representação de probabilidade relativa baseada apenas na confiança contida nas próprias correlações do modelo. Isso não é a probabilidade do dado ser certo no mundo real, mas a discrepância na balança dos pesos da rede. Se a nova imagem for completamente fora da distribuição dos dados originais (por erro de escala ou cor), ela ativará certos neurônios desproporcionalmente, gerando um colapso matemático que o sistema interpretará como absoluta (e cega) "certeza".


# Taxa de Aprendizado e Função de Perda

In [ ]:
# Construir modelo com nova arquitetura e taxa de aprendizado maior

model_lr = keras.Sequential([
    layers.Input(shape=(784,)),                 # Dimensionalidade dos dados de entrada
    layers.Dense(128, activation="relu"),       # 1ª camada oculta: 128 neurônios com ReLU
    layers.Dense(64, activation="tanh"),         # 2ª camada oculta: 64 neurônios com Tanh
    layers.Dense(10, activation="softmax")       # Saída: Softmax
])

# Configurar processo de treinamento

model_lr.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),   # Taxa de aprendizado maior
    loss="sparse_categorical_crossentropy",                 # Função de perda
    metrics=["accuracy"]                                    # Métrica de avaliação
)

# Treinar modelo

historico_lr = model_lr.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.20,
    verbose=1
)

## Perguntas
21. Qual hiperparâmetro de treinamento foi alterado em relação ao modelo de referência e qual foi a alteração realizada?

R.: O hiperparâmetro de treinamento alterado foi a taxa de aprendizado (learning_rate) do otimizador Adam. Ela foi aumentada de 0.001 (no modelo de referência) para 0.01.  


22. Comparando os dois treinamentos, que efeito o aumento da taxa de aprendizado produziu sobre o comportamento da loss e da accuracy?

R.: O aumento da taxa de aprendizado gerou maior instabilidade durante o treinamento. Observando o histórico, a perda de validação (val_loss) no modelo com taxa maior flutuou bastante (caindo e subindo entre as épocas) em vez de diminuir de forma suave, e a precisão (accuracy) também apresentou oscilações, indicando que o modelo teve mais dificuldade para convergir de forma estável.  


23. Quais hiperparâmetros de arquitetura também foram alterados em relação ao modelo de referência? Identifique as mudanças realizadas na quantidade de camadas ocultas, no número de neurônios e na função de ativação.

R.: Em relação ao modelo de referência (que tinha apenas 1 camada oculta), a nova arquitetura passou a ter duas camadas ocultas. A primeira camada foi mantida com 128 neurônios e função de ativação ReLU, mas foi adicionada uma segunda camada oculta com 64 neurônios e função de ativação Tanh.

# Otimizador

In [ ]:
# Construir modelo com a mesma arquitetura e outro otimizador

model_sgd = keras.Sequential([
    layers.Input(shape=(784,)),                 # Dimensionalidade dos dados de entrada
    layers.Dense(128, activation="relu"),       # 1ª camada oculta: 128 neurônios com ReLU
    layers.Dense(64, activation="tanh"),        # 2ª camada oculta: 64 neurônios com Tanh
    layers.Dense(10, activation="softmax")      # Saída: Softmax
])

# Configurar processo de treinamento

model_sgd.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),    # Alterar apenas o otimizador
    loss="sparse_categorical_crossentropy",                # Manter função de perda
    metrics=["accuracy"]                                   # Métrica de avaliação
)

# Treinar modelo

historico_sgd = model_sgd.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.20,
    verbose=1
)

## Perguntas
24. Qual hiperparâmetro foi alterado em relação ao modelo anterior e qual foi a alteração realizada?

R.: O hiperparâmetro alterado foi o otimizador. A alteração realizada foi a substituição do otimizador Adam pelo otimizador SGD (Stochastic Gradient Descent).


25. Mantendo a mesma arquitetura, a mesma taxa de aprendizado, a mesma função de perda, o mesmo número de épocas e o mesmo tamanho de lote, que diferenças podem ser observadas no treinamento com Adam e SGD?

R.: O otimizador Adam propiciou um aprendizado inicial muito mais rápido, atingindo 80% de acurácia já na primeira época e terminando o treinamento com uma perda (loss) de 0.3256. Já o SGD demonstrou uma curva de aprendizado bem mais lenta e gradual, iniciando a primeira época com apenas 62,65% de acurácia e encerrando a 20ª época com uma perda (loss) maior, de 0.3914, indicando que o SGD precisa de mais tempo (ou épocas) para alcançar o mesmo nível de convergência do Adam nessas configurações.  

26. Considerando todo o laboratório, explique a diferença entre hiperparâmetros de arquitetura e hiperparâmetros de treinamento, citando exemplos utilizados nos modelos construídos.

R.:

  Hiperparâmetros de arquitetura: Definem a estrutura da rede neural, ou seja, como o modelo é construído antes mesmo de qualquer aprendizado acontecer. Eles determinam a complexidade e a capacidade do modelo de extrair padrões dos dados.  Exemplos utilizados nos modelos: A dimensão da camada de entrada (shape=(784,)), o número de camadas ocultas (os testes variaram entre 1 e 2 camadas ocultas), a quantidade de neurônios dentro dessas camadas (128, 64 e 10) e as funções de ativação escolhidas para cada uma delas (ReLU, Tanh e Softmax). Exemplos citados no material: O número de camadas ocultas, o número de neurônios por camada oculta e a escolha das funções de ativação (como Sigmoid, Tanh, ReLU na camada oculta, ou Softmax/Identidade na camada de saída)

  Hiperparâmetros de treinamento: Controlam a dinâmica do processo de aprendizado, ditando a forma e a velocidade com que os pesos (parâmetros treináveis) da rede são atualizados durante a fase de otimização.  Exemplos utilizados nos modelos: O algoritmo otimizador selecionado (Adam ou SGD), a taxa de aprendizado ou learning rate (0.001 ou 0.01), a função de perda a ser minimizada (sparse_categorical_crossentropy), a quantidade de épocas de treinamento (20), o tamanho do lote ou batch size (128) e a proporção dos dados separada para validação (validation_split=0.20). Exemplos citados no material: O tamanho do lote (batch size), o número de épocas (epochs), a função de perda (loss function, como MSE ou Cross-Entropy), a escolha do otimizador (como SGD, RMSprop ou Adam) e a taxa de aprendizado (learning rate).